### Building an Agentic AI + MPI Loop with DragonHPC

**Estimated time:** ~30 minutes  
**Format:** 5 exercises. Each includes background, a small coding task, and a hidden solution.

---

## Session goals

This notebook walks you through the building blocks behind the two agentic
workflows in this directory,
[lightweight_agent_workflow.py](lightweight_agent_workflow.py) and
[ultralight_agent_workflow.py](ultralight_agent_workflow.py). Both search for
the largest stable CFL (Courant number) for a small MPI CFD solver
([../../course2/orchestrating_MPI/mpi4py_example.py](../../course2/orchestrating_MPI/mpi4py_example.py))
by coupling an MPI application to LLM agents through a shared Distributed
Dictionary (DDict).

You will practice how to:

- launch an MPI application with a Dragon `Batch` and a `ProcessTemplate`
- share data between the MPI ranks and Python via a `DDict`
- scan the shared `DDict` for NaNs produced by an unstable simulation
- compose LLM agents into a `Pipeline` driven by the `DAGOrchestrator`
- chain deterministic function nodes into a pipeline
- run the full workflows end to end

The exercises build on one another: Exercise 1 writes data, Exercise 2 reads
it back, Exercises 3 and 4 introduce the pipeline abstraction, and Exercise 5
runs the complete workflows.


## Setup - run this first

Start Jupyter from a Dragon-enabled environment (for example with
`dragon-jupyter`) and run the next cell. It imports Dragon, sets the
multiprocessing start method to `"dragon"`, and defines a few constants shared
by all exercises.

Exercises 3 and 5 (the LLM parts) need the local model in the repository's
`model/` directory. Exercises 1, 2, and 4 are deterministic and do not need a
model or a GPU.


In [1]:
import os
import sys
import math
from functools import partial
from pathlib import Path
from typing import Callable

import dragon
import multiprocessing as mp
import torch
try:
    mp.set_start_method("dragon")
except RuntimeError:
    pass

from dragon.data.ddict import DDict
from dragon.native.process import ProcessTemplate
from dragon.infrastructure.facts import PMIBackend
from dragon.workflows.batch import Batch

# Imports necessary for exercise 3 and 4
from dragon.ai.agent.core import create_sub_agent
from dragon.ai.agent.config import (
    AgentConfig,
    OrchestratorConfig,
    Pipeline,
    PipelineNode,
    TaskResult,
    TaskStatus,
    DISPATCH_ID_KEY,
    RESULT_KEY,
    STATUS_KEY,
)
from dragon.ai.agent.tools import ToolRegistry
from dragon.ai.agent.orchestrator import DAGOrchestrator
from dragon.native.event import Event
from dragon.native.process import Process
from dragon.native.queue import Queue
from inference_utils import lightweight_inference_service

# Number of MPI ranks each CFD job runs on.
NUM_RANKS = 4

# The MPI CFD solver used throughout the notebook (relative to this notebook).
MPI_APP = "../../course2/orchestrating_MPI/mpi4py_example.py"

# Location of the local model (only needed for the LLM exercises 3 and 5).
# If you have the model downloaded somewhere other than the root dir of the repo, you can set the environment variable DRAGON_LOCAL_MODEL_DIR to point to it.  Otherwise, it will default to the SmolLM3_3B directory in the root of the repo.
_REPO_ROOT = Path.cwd().resolve().parents[1]
LOCAL_MODEL_DIR = os.environ.get(
    "DRAGON_LOCAL_MODEL_DIR", str(_REPO_ROOT / "SmolLM3_3B")
)

print("Dragon + PyTorch ready")
print("MPI app exists:", (Path.cwd() / MPI_APP).resolve().exists())
print("Model dir exists:", Path(LOCAL_MODEL_DIR).exists())
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

Dragon + PyTorch ready
MPI app exists: True
Model dir exists: True
CUDA available: False
GPU count: 0


In [ ]:
# Variables that can impact performance when running the LLM on a CPU
torch.set_num_threads(8)
torch.set_num_interop_threads(1)
!export OMP_PROC_BIND='close'
!export OMP_PLACES='cores'

---

## Exercise 1 - Run an MPI job with a `Batch` and share a `DDict`

**Background:**

The CFD solver in
[../../course2/orchestrating_MPI/mpi4py_example.py](../../course2/orchestrating_MPI/mpi4py_example.py)
is an mpi4py program. Each rank advects a square wave and, at the end, writes
its slice of the solution into a shared `DDict`:

```python
# inside mpi4py_example.py
if ddict is not None:
    ddict[f"cfl_{cfl}_rank{rank}"] = u[2:-2]
```

Dragon's `Batch` launches MPI jobs. You describe the ranks with a
`ProcessTemplate` (here, the Python interpreter running the solver) and submit
them with `batch.options(pmi=...).job(process_templates=...)`. The solver reads
the serialized `DDict` from the `--dser` command-line argument and attaches to
it, so the ranks and your notebook share the same dictionary. This is the same
pattern used by `queue_job` in [ai_cfd_workflow.py](ai_cfd_workflow.py).

Demo pattern:

```python
rank_tmpls = [(nranks, ProcessTemplate(
    target=sys.executable,
    args=(MPI_APP, "--cfl", str(cfl), "--dser", ddict.serialize()),
    stdout="/dev/null",
))]
job = batch.options(pmi=PMIBackend.PMIX).job(process_templates=rank_tmpls)
ecodes = job.get()   # blocks until the MPI job finishes
```

**Your task:**

1. Write `run_cfd_batch(batch, nranks, cfl, ddict)` that builds a
   `ProcessTemplate` for the MPI solver and submits it to the `Batch`.
2. Pass the serialized `ddict` so every rank writes its result slice into it.
3. Run one job on `NUM_RANKS` ranks **with a CFL of 3** and wait for it to
   finish.

Keep the `DDict` (`ex1_ddict`) alive - Exercise 2 reads from it.


In [ ]:
# -- Exercise 1 -- your code here ---------------------------------------------

def run_cfd_batch(batch, nranks, cfl, ddict):
    pass

ecodes: [0, 0, 0, 0]
keys: ['cfl_3.0_rank3', 'cfl_3.0_rank1', 'cfl_3.0_rank2', 'cfl_3.0_rank0']


In [ ]:
# -- Exercise 1 -- test your code here -----------------------------------------
ex1_ddict = DDict(1, 1, 64 * 1024 * 1024)
ex1_batch = Batch()

job = run_cfd_batch(ex1_batch, NUM_RANKS, 3.0, ex1_ddict)
print("ecodes:", job.get())          # blocks until the MPI job completes

# The ranks wrote keys like "cfl_3.0_rank0" .. "cfl_3.0_rank3".
print("keys:", list(ex1_ddict.keys()))

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def run_cfd_batch(batch, nranks, cfl, ddict):
    # One template covers all ranks: the Python interpreter runs the mpi4py
    # solver, receiving the CFL and the serialized DDict on the command line.
    rank_tmpls = [(nranks, ProcessTemplate(
        target=sys.executable,
        args=(MPI_APP, "--cfl", str(cfl), "--dser", ddict.serialize()),
        stdout="/dev/null",
    ))]
    # PMIx wires up the MPI ranks; .job() submits them to the Batch.
    job = batch.options(pmi=PMIBackend.PMIX).job(process_templates=rank_tmpls)
    return job
```

CFL 3 is well above the solver's stability limit, so this run is expected to
blow up - Exercise 2 detects that.

</details>


---

## Exercise 2 - Scan the `DDict` for NaNs

**Background:**

When a solver runs above its stability limit it produces `NaN`s or values that
blow up toward infinity. The agentic workflows use a "NaN checker" to decide
which CFL values were stable. The core check (see `_check_nans` in
[lightweight_agent_workflow.py](lightweight_agent_workflow.py)) treats a rank
array as bad if it contains a `NaN` **or** if any value grows beyond
`+/- 1e6`.

In Exercise 1 the MPI ranks wrote one array per rank under keys like
`cfl_3.0_rank0`. Here you iterate over exactly those keys and report which
ranks blew up.

**Your task:**

1. Write `check_values(values)` that returns `True` if the array has a NaN or
   an out-of-range value.
2. Write `scan_for_nans(ddict)` that iterates over the keys written in
   Exercise 1 and returns a `{key: has_nans}` report.
3. Run it against `ex1_ddict` and print the report.


In [ ]:
# -- Exercise 2 -- your code here ---------------------------------------------

def check_values(values):
    pass

def scan_for_nans(ddict):
    pass

cfl_3.0_rank3: stable
cfl_3.0_rank1: NaN / blew up
cfl_3.0_rank2: stable
cfl_3.0_rank0: NaN / blew up


In [5]:
# Done with Exercise 1 and 2's resources - clean them up.
ex1_batch.join()
ex1_ddict.destroy()

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def check_values(values):
    for v in values:
        try:
            if math.isnan(float(v)):
                return True
        except (TypeError, ValueError):
            # None, strings, etc. can't be a valid number -> treat as NaN-like.
            return True
    # An explicit solver above its CFL limit grows without bound.
    return max(values) > 1e6 or min(values) < -1e6


def scan_for_nans(ddict):
    report = {}
    for key in ddict.keys():
        # Only look at the per-rank result arrays the MPI job wrote.
        if key.startswith("cfl_") and "_rank" in key:
            report[key] = check_values(list(ddict[key]))
    return report


report = scan_for_nans(ex1_ddict)
for key, bad in report.items():
    print(f"{key}: {'NaN / blew up' if bad else 'stable'}")
```

Because Exercise 1 ran at CFL 3, at least one rank should report `NaN / blew up`. 

</details>


---

## Exercise 3 - Chain Exercises 1 and 2 in a `Pipeline`

**Background:**
The workflows compose LLM agents into a `Pipeline` executed by the
`DAGOrchestrator`. Each `PipelineNode` names an agent and declares its
`depends_on` edges; the orchestrator passes an upstream node's output as the
message to the downstream node. A `PipelineNode` can also run a plain Python function (`fn=...`) instead of an LLM agent. This is exactly what
[ultralight_agent_workflow.py](ultralight_agent_workflow.py) does: every node
is a deterministic function that reads/writes the shared `DDict` and publishes
its result into the orchestrator's `DDict` so downstream nodes can run.

A function node has the signature `fn(*upstreams: TaskResult) -> TaskResult`. To make it easier to generate such functions we have provided the `pipeline_function_wrapper` below. We also have provided a `run_pipeline` function that takes a `batch`, a `prompt` that gets passed to the workflow, a `pipeline`, and optionally a list of `agent_specs`. 

**Your task:**

Build a two-node pipeline that runs Exercise 1 then Exercise 2 back to back:

1. `run_experiments` (root) - runs the CFD `Batch` (reuse `run_cfd_batch`) at
   CFL 3 and publishes what it ran result.
2. `check_nans` (`depends_on=["run_experiments"]`) - scans the `DDict`
   (reuse `scan_for_nans`) and publishes the NaN report.

Each function should return it's `agent_id` and it's report of the run in that order. Run your pipeline with the `run_pipeline` function (with `agents=[]`, since there are no LLM agents) and print the final report. Remember to wrap each function in the `pipeline_function_wrapper` using a partial after creating a `DDict` and `Batch`. 


In [4]:
def pipeline_function_wrapper(fn: Callable, args, kwargs, *upstreams):
    """Common plumbing shared by every deterministic function node.

    A function node only needs to do its own work; the DAG bookkeeping is
    identical everywhere, so it lives here once. For more complex nodes, this simpler wrapper might need to be replaced with a more sophisticated one that handles multiple upstreams, etc.
    """
    # The orchestrator seeds the root node with a TaskResult carrying the
    # task_id and the serialized orchestrator DDict.
    upstream = upstreams[0]
    task_id = upstream.task_id
    serialized_ddict = upstream.serialized_ddict

    ddict = DDict.attach(serialized_ddict)  # orchestrator's result DDict
    try:
        # The node does only its own work and tells us who it is + what to say.
        agent_id, response = fn(*args, **kwargs)

        own_id = f"fn-{agent_id}-{task_id[:8]}"
        ddict[DISPATCH_ID_KEY.format(task_id=task_id, agent_id=agent_id)] = own_id
        ddict[RESULT_KEY.format(task_id=task_id, agent_id=agent_id, dispatch_id=own_id)] = {
            "response": response
        }
        ddict[STATUS_KEY.format(task_id=task_id, agent_id=agent_id, dispatch_id=own_id)] = TaskStatus.DONE
    finally:
        ddict.detach()

    return TaskResult(
        task_id=task_id,
        agent_id=agent_id,
        status=TaskStatus.DONE,
        serialized_ddict=serialized_ddict,
    )

In [4]:
def run_pipeline(prompt: str, batch: Batch, pipeline: Pipeline,  agent_specs: list = []):
    """Run `pipeline` on a `DAGOrchestrator` once and tear everything down.

    Shared driver for both kinds of pipeline in this notebook:

    * ``pipeline``  -> a deterministic function pipeline (Exercise 3) or a pipeline of agents.
    * ``agent_specs=[...]`` -> an LLM-agent pipeline (Exercise 4). Each spec is
      the dict ``create_sub_agent`` expects (``config`` built with the shared
      ``inference_queue``, plus ``tool_registry``, ``shutdown_event`` and
      ``reply_queue``). The shared inference service is started, every agent is
      launched as its own Dragon ``Process``, and all of them are shut down in
      the ``finally``.
    """
    use_agents = len(agent_specs) > 0

    # Start the shared inference service (the local model) only when there are
    # agents to serve. Its lifecycle is owned entirely by this function.
    inference_proc = None
    inference_shutdown = None
    if use_agents:
        inference_shutdown = Event()
        inference_proc = Process(
            target=lightweight_inference_service,
            args=(agent_specs[0]["config"].inference_queue, inference_shutdown, LOCAL_MODEL_DIR),
        )
        inference_proc.start()

    procs = []
    orchestrator = None
    try:
        if use_agents:
            # Launch each agent as its own Dragon Process...
            for spec in agent_specs:
                p = Process(target=create_sub_agent, kwargs=spec)
                p.start()
                procs.append(p)
            # ...and collect the input queue each agent publishes once ready.
            for spec in agent_specs:
                spec["config"].input_queue = spec["reply_queue"].get()

        orchestrator = DAGOrchestrator(
            config=OrchestratorConfig(
                agents=[s["config"] for s in agent_specs],
                poll_interval=2,
                poll_timeout=1200.0,
            ),
            pipeline=pipeline,
        )
        result = orchestrator.run(user_input=prompt, batch=batch)
        return result.get("response", str(result)) if isinstance(result, dict) else str(result)
    finally:
        if orchestrator is not None:
            orchestrator.destroy()
        batch.join()
        # Shut the agents down, then the inference service they depend on.
        for spec in agent_specs:
            spec["shutdown_event"].set()
        for p in procs:
            p.join()
        if inference_shutdown is not None:
            inference_shutdown.set()
        if inference_proc is not None:
            inference_proc.join()

In [ ]:
# -- Exercise 3 -- your code here ---------------------------------------------

def run_experiments_fn(batch, data_store, nranks, cfl):
    pass

def check_nans_fn(data_store, nranks):
    pass

def create_pipeline():
    pass

[fn] check_nans -> cfl_3.0_rank0: NaN, cfl_3.0_rank1: NaN, cfl_3.0_rank2: stable, cfl_3.0_rank3: stable

Final report:
cfl_3.0_rank0: NaN, cfl_3.0_rank1: NaN, cfl_3.0_rank2: stable, cfl_3.0_rank3: stable


In [ ]:
# -- Exercise 3 -- test your code here -----------------------------------------
data_store = None
try:
    data_store = DDict(1, 1, 2 * 1024 * 1024)
    batch = Batch(managed_lifecycle=True, results_ddict_mem=int(10 * 1024 * 1024))
    # The partial of the pipeline function wrapper passes args and kwargs dict to the function and wraps the function with the orchestrator's DAG bookkeeping.
    partial_run_experiments_fn = partial(pipeline_function_wrapper, run_experiments_fn, (batch, data_store, NUM_RANKS, 3.0), {})
    partial_check_nans_fn = partial(pipeline_function_wrapper, check_nans_fn, (data_store, NUM_RANKS), {})

    pipeline = create_pipeline()

    # No agents -> deterministic pipeline; run_pipeline owns the orchestrator
    # and its Batch.
    report = run_pipeline("Run the CFD job and check for NaNs.", batch, pipeline)
    print("\nFinal report:")
    print(report)
finally:
    if data_store is not None:
        data_store.destroy()
    if batch is not None:
        batch.join()
        batch.destroy(force_timeout=1)

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def run_experiments_fn(batch, data_store, nranks, cfl):
    # Node-specific work only: run the MPI CFD job into the shared data store.
    # A self-contained Batch launches the job (exactly like Exercise 1); the
    # DAG bookkeeping lives in pipeline_function_wrapper.
    try:
        job = run_cfd_batch(batch, nranks, cfl, data_store)
        job.get()  # block until the MPI job finishes

    return "run_experiments", f"Ran CFD at CFL={cfl}."


def check_nans_fn(data_store, nranks):
    # Node-specific work only: scan the shared data store for NaNs.
    report = scan_for_nans(data_store)
    text = ", ".join(
        f"{k}: {'NaN' if bad else 'stable'}" for k, bad in report.items()
    )
    print(f"[fn] check_nans -> {text}", flush=True)

    return "check_nans", text

def create_pipeline():
    pipeline = Pipeline(nodes=[
        PipelineNode(
            agent_id="run_experiments",
            fn=partial_run_experiments_fn,
            depends_on=[],
        ),
        PipelineNode(
            agent_id="check_nans",
            fn=partial_check_nans_fn,
            depends_on=["run_experiments"],
        ),
    ])
    return pipeline

```

</details>

---

## Optional Exercise 4 - A two-agent `Pipeline` (answerer + critic)

**Background:**

As discussed above, LLM agents can also be nodes of a `Pipeline` executed by the
`DAGOrchestrator`. For agentic nodes, the `PipelineNode` is expected to have a task description that is given to the agent as well as any tools the agent is allowed to call. In [lightweight_agent_workflow.py](lightweight_agent_workflow.py), we define a `nan_agent` and `generator_agent` that are used in a loop to search for the CFL condition and are allowed to use tools like the `scan_for_nans` that you wrote above. 

Here you build a simple **two-agent** pipeline where **neither agent uses
tools** (each `ToolRegistry` is left empty):

- `answerer` - answers the user's question in plain text.
- `critic` - depends on `answerer` and critiques that answer.

The agents are served by the local model through
`lightweight_inference_service` from [inference_utils.py](inference_utils.py),
launched as its own Dragon `Process` and shared via an inference `Queue`. Each
agent is started with `create_sub_agent` and publishes its input queue back on
a `reply_queue` once it is ready.

**Your task:**

1. Define how the answer should answer your question. If on limited hardware we suggest that you explicitly limit the length to no more than a couple of sentences. 
2. Define how the critiquer should critic the answer. 
3. Give the agents a prompt.

> This exercise needs the local model in `SmolLM3_3B/`. Skip it if you do not have
> the model or enough RAM/GPU.


In [ ]:
# -- Optional Exercise 4 -- your code here -------------------------------------
answer_task_description = ("") # Provide a prompt for how you want the agent to answer the question.
critique_task_description = ("") # Provide a prompt for how you want the agent to critique the question.
prompt = "" # Ask your question here. The answerer will answer it, and the critic will critique the answer.

[inference] Loading model from /workspaces/pearc_agenticloop_clean/SmolLM3_3B on cpu...


Loading weights: 100%|██████████| 326/326 [00:27<00:00, 11.77it/s]


[inference] Model ready — serving requests.



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[inference] payload='A CFL condition, or Courant-Friedrichs-Lewy condition, is a criterion used in numerical methods to ensure stability and accuracy in solving partial differential equations, particularly in time-stepping schemes. It requires the time step to be small enough relative to the spatial step to prevent numerical instability.'
[inference] payload='The answer is clear and concise, effectively explaining the CFL condition and its purpose. However, it could benefit from a more formal and technical tone, as it is addressing a specialized topic in numerical methods. Additionally, providing a brief example or formula for the CFL condition might enhance understanding for readers unfamiliar with the concept.'
[inference] Shutdown signalled — inference service stopping.


In [ ]:
# -- Optional Exercise 4 -- test your code here --------------------------------
input_queue = Queue()

# The pipeline: answerer -> critic. Neither node has a tool.
pipeline = Pipeline(nodes=[
    PipelineNode(
        agent_id="answerer",
        task_description=answer_task_description,
        depends_on=[],
    ),
    PipelineNode(
        agent_id="critic",
        task_description=critique_task_description,
        depends_on=["answerer"],
    ),
])

# One AgentConfig + empty ToolRegistry per agent (empty registry = no tools).
# Every config shares the same inference_queue so run_pipeline can start the
# single inference service that serves them.
agent_specs = [
    {
        "config": AgentConfig(
            agent_id="answerer",
            name="Answerer",
            role="You answer questions in plain text. You never call tools.",
            inference_queue=input_queue,
            max_concurrent_requests=1,
            max_tool_call_iterations=2,
        ),
        "tool_registry": ToolRegistry(),
        "shutdown_event": Event(),
        "reply_queue": Queue(),
    },
    {
        "config": AgentConfig(
            agent_id="critic",
            name="Critic",
            role="You critique answers in plain text. You never call tools.",
            inference_queue=input_queue,
            max_concurrent_requests=1,
            max_tool_call_iterations=2,
        ),
        "tool_registry": ToolRegistry(),
        "shutdown_event": Event(),
        "reply_queue": Queue(),
    },
]

# Passing agent_specs makes run_pipeline start + manage the inference
# service and one Process per agent for us.
batch = None
try:
    batch = Batch(managed_lifecycle=True, results_ddict_mem=int(10 * 1024 * 1024))
    #batch = Batch(results_ddict_mem=int(10 * 1024 * 1024))
    run_pipeline(prompt, batch, pipeline, agent_specs)
finally:
    if batch is not None:
        batch.join()
        batch.destroy(force_timeout=1)

<details>
<summary><b>▶ Show Solution</b></summary>

```python
answer_task_description = ("Provide a clear and concise answer to the question. Do not answer with more than two sentences.") # Provide a prompt for how you want the agent to answer the question.
critique_task_description = ("Critique the answer. Provide your critique in a clear and concise way. Do not answer with more than two sentences") # Provide a prompt for how you want the agent to critique the question.
prompt = "What is a CFL condition?" # Ask your question here. The answerer will answer it, and the critic will critique the answer.

```

</details>

---

## Optional Exercise 5 - Run the full workflows

**Background:**

You now have all the pieces the two shipped workflows are built from. Their
`__main__` blocks wire a NaN checker, a CFL generator, and an experiment runner
into a `Pipeline` and iterate the CFL search:

- [ultralight_agent_workflow.py](ultralight_agent_workflow.py) - **fully
  deterministic** Every node is a function. This is
  the one to run if you have limited compute.
- [lightweight_agent_workflow.py](lightweight_agent_workflow.py) - the **LLM
  version**: a `nan_agent` and a `generator_agent` driven by the local model
  through `lightweight_inference_service`. Needs the `SmolLM3_3B/` directory and
  more RAM/GPU.

Both expose a `run(init_cfls, num_ranks, iterations, user_prompt)` function, so
you can call them directly from the notebook.

**Your task:**

Pick the workflow that matches your compute resources and run it. Start with
the ultralight (deterministic) one. Examine the one you run and see how the APIs you used above can help write an agentic workflow!


In [ ]:
# -- Exercise 5 -- your code here ---------------------------------------------

prompt = (
    "Check the latest CFD result arrays for NaNs and report which "
    "CFL values were stable and which produced NaNs. "
    "Choose the next set of CFL values, increasing CFL for stable ranks "
    "and decreasing it for ranks that produced NaNs."
)
num_ranks = NUM_RANKS
iterations = 2
init_cfls=[0.1, 1.2, 3.6, 8.7]
# Option A -- deterministic, no model required, less compute intensive. Makes decisions independent of the prompt:
import ultralight_agent_workflow as workflow
# Option B -- LLM agents, relies on the local model from course 2, more compute intensive. Agents take prompt into consideration when deciding what to do:
#import lightweight_agent_workflow as workflow
workflow.run(
     init_cfls=init_cfls,
     num_ranks=num_ranks,
     iterations=iterations,
     user_prompt=prompt,
)

[startup] Launching lightweight inference service...
[startup] Agent 'generator_agent' ready.
[startup] Agent 'nan_agent' ready.

[fn] run_experiments_node -> cfl_values=[0.1, 1.2, 3.6, 8.7]
Got a job <dragon.workflows.batch.batch.Job object at 0xffff94568770>
Got a job <dragon.workflows.batch.batch.Job object at 0xffff946d7680>
Got a job <dragon.workflows.batch.batch.Job object at 0xffff946d6f00>
Got a job <dragon.workflows.batch.batch.Job object at 0xffff946d7650>
Rank 1 of 4 says: Hello from MPI!
Using CFL=0.1 and provided DDict=<dragon.data.ddict.ddict.DDict object at 0xffffb0aa4b30>
Rank 3 of 4 says: Hello from MPI!
Using CFL=0.1 and provided DDict=<dragon.data.ddict.ddict.DDict object at 0xffffa7554b30>
Rank 2 of 4 says: Hello from MPI!
Using CFL=0.1 and provided DDict=<dragon.data.ddict.ddict.DDict object at 0xffff92904b30>
Rank 0 of 4 says: Hello from MPI!
Using CFL=0.1 and provided DDict=<dragon.data.ddict.ddict.DDict object at 0xffff7f704b30>
Step =      1 : SimTime = 9.7656e

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

Rank 0 of 4 says: Hello from MPI!
Rank 2 of 4 says: Hello from MPI!
Using CFL=3.6 and provided DDict=<dragon.data.ddict.ddict.DDict object at 0xffffa7fc4b30>
Using CFL=3.6 and provided DDict=<dragon.data.ddict.ddict.DDict object at 0xffffb72a4b30>
Rank 3 of 4 says: Hello from MPI!
Using CFL=3.6 and provided DDict=<dragon.data.ddict.ddict.DDict object at 0xffff86164b30>
Rank 1 of 4 says: Hello from MPI!
Using CFL=3.6 and provided DDict=<dragon.data.ddict.ddict.DDict object at 0xffffa0924b30>
Step =      1 : SimTime = 3.5156e-03 : StepElapsed = 7.66e-02 s : CellsPS = 1.34e+04
Step =      2 : SimTime = 7.0313e-03 : StepElapsed = 1.83e-02 s : CellsPS = 5.60e+04
Step =      3 : SimTime = 1.0547e-02 : StepElapsed = 2.91e-04 s : CellsPS = 3.52e+06
Step =      4 : SimTime = 1.4063e-02 : StepElapsed = 1.54e-03 s : CellsPS = 6.63e+05
Step =      5 : SimTime = 1.7578e-02 : StepElapsed = 7.35e-04 s : CellsPS = 1.39e+06
Step =      6 : SimTime = 2.1094e-02 : StepElapsed = 2.51e-04 s : CellsPS = 4.0

Loading weights:  15%|█▌        | 49/326 [00:11<00:44,  6.25it/s]

Got ecodes: [0, 0, 0, 0]
Iteration 1/2
Request: Check the latest CFD result arrays for NaNs and report which CFL values were stable and which produced NaNs. Choose the next set of CFL values, increasing CFL for stable ranks and decreasing it for ranks that produced NaNs.



...

[inference] Model ready — serving requests.



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this

[inference] payload='{"response": {"type": "tool_request", "tool_calls": [{"name": "scan_all_ranks", "args": {}}]}}'


 warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[tool] Result for key: cfl_0.1_rank0: {'has_nans': np.False_, 'max_value': np.float64(1.0), 'min_value': np.float64(0.0)}
[tool] Result for key: cfl_0.1_rank1: {'has_nans': np.False_, 'max_value': np.float64(1.0), 'min_value': np.float64(-3.6393617756172457e-69)}
[tool] Result for key: cfl_0.1_rank2: {'has_nans': np.False_, 'max_value': np.float64(0.0), 'min_value': np.float64(0.0)}
[tool] Result for key: cfl_0.1_rank3: {'has_nans': np.False_, 'max_value': np.float64(0.0), 'min_value': np.float64(0.0)}
[tool] Result for key: cfl_1.2_rank0: {'has_nans': np.True_, 'max_value': np.float64(227416471796.08408), 'min_value': np.float64(-225675200342.9062)}
[tool] Result for key: cfl_3.6_rank0: {'has_nans': np.True_, 'max_value': np.float64(7.055475068126655e+21), 'min_value': np.float64(-7.133663490311874e+21)}
[tool] Result for key: cfl_8.7_rank0: {'has_nans': np.True_, 'max_value': np.float64(44530667428613.1), 'min_value': np.float64(-38978054683506.945)}
[inference] payload='{"response":